# Smoke test — tiny CPU regression fine-tune
Checks the whole fine-tuning path end to end (selection, DSLab execution, progress in Chat, MLflow
logging, output collection) in about a minute, with no GPU and only the Python standard library.
It "fine-tunes" a linear model `y = w*x + b` by gradient descent on synthetic data (or a CSV with
`x,y` columns) and saves the model and metrics to the run's output folder.

Use it to verify a new DSLab runtime or MLflow connection before running real GPU jobs.

## Parameters

In [ ]:
epochs = 30              # training epochs
learning_rate = 0.05     # gradient-descent step size
n_samples = 200          # synthetic samples when no dataset is given
dataset_path = ""        # optional CSV with x,y columns
sleep_per_epoch = 1.0    # seconds per epoch, to make progress visible
seed = 42                # random seed
output_dir = "outputs"   # where the model and metrics are written

## Setup

In [ ]:
import csv, json, math, os, random, sys, time
output_dir = os.environ.get("STUDIO_OUTPUT_DIR", output_dir) if output_dir == "outputs" else output_dir
os.makedirs(output_dir, exist_ok=True)
random.seed(seed)
try:
    import mlflow
    HAVE_MLFLOW = bool(os.environ.get("MLFLOW_RUN_ID"))
except Exception:
    HAVE_MLFLOW = False
print("python", sys.version.split()[0], "| mlflow logging:", HAVE_MLFLOW, "| output:", output_dir)

## Load data

In [ ]:
if dataset_path:
    with open(dataset_path) as f:
        rows = [(float(r["x"]), float(r["y"])) for r in csv.DictReader(f)]
else:
    rows = [(x, 3.0 * x + 0.5 + random.gauss(0, 0.1)) for x in (random.uniform(-1, 1) for _ in range(n_samples))]
random.shuffle(rows)
split = int(len(rows) * 0.8)
train, val = rows[:split], rows[split:]
print(f"train={len(train)} val={len(val)}")

## Train

In [ ]:
def mse(data, w, b):
    return sum((w * x + b - y) ** 2 for x, y in data) / max(1, len(data))

w, b = 0.0, 0.0
if HAVE_MLFLOW:
    mlflow.start_run()            # resumes the run the platform created (MLFLOW_RUN_ID)
    mlflow.log_params({"epochs": epochs, "learning_rate": learning_rate, "n_train": len(train)})
for epoch in range(1, epochs + 1):
    gw = sum(2 * (w * x + b - y) * x for x, y in train) / len(train)
    gb = sum(2 * (w * x + b - y) for x, y in train) / len(train)
    w, b = w - learning_rate * gw, b - learning_rate * gb
    loss, val_loss = mse(train, w, b), mse(val, w, b)
    if HAVE_MLFLOW:
        mlflow.log_metrics({"loss": loss, "eval_loss": val_loss, "epoch": epoch}, step=epoch)
    print("##STUDIO_PROGRESS " + json.dumps({"step": epoch, "total_steps": epochs, "loss": round(loss, 5),
                                              "eval_loss": round(val_loss, 5)}))
    print(f"epoch {epoch}/{epochs} loss={loss:.5f} val_loss={val_loss:.5f}")
    time.sleep(sleep_per_epoch)

## Save model and metrics

In [ ]:
metrics = {"train_loss": mse(train, w, b), "eval_loss": mse(val, w, b), "w": w, "b": b}
with open(os.path.join(output_dir, "model.json"), "w") as f:
    json.dump({"type": "linear", "w": w, "b": b}, f)
with open(os.path.join(output_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=1)
if HAVE_MLFLOW:
    mlflow.log_metrics({"final_eval_loss": metrics["eval_loss"]})
    mlflow.end_run()
print(json.dumps(metrics, indent=1))